# 3D-VGA — step 2: speech → encoder features

Runs the frozen speech encoder (wav2vec 2.0 base by default) once over every WAV and stores its frames at 25 per second, with six prosody channels computed from the waveform. Built for **Run All**. Needs a GPU for the full corpus.

Reads `data_audio/wav/` from `run_audio_tts.ipynb`; writes `data_audio/feat/<encoder>/<split>.h5`, the only audio file training reads. Next: `run_audio_train_eval.ipynb`.

## 1 — configuration

The only cell you need to edit.

In [ ]:
# USER CONFIGURATION

DEVICE     = "auto"      # "auto" | "cpu" | "cuda"   -- auto probes for a working GPU
GPU_ID     = "0"
QUICK_TEST = False       # True: 2 scenes per split into data_audio/_quick -- minutes, and
                         #       never mixed with the real corpus

SPLITS = ["train", "val"]

RUN_ENCODE        = True
ENCODER           = "wav2vec2"         # wav2vec2 | wav2vec2-large | wavlm | hubert
ENCODER_LAYER     = -1                 # hidden layer read; -1 = the encoder's default
                                       # (8 of 12; 16 of 24 for wav2vec2-large)
MAX_AUDIO_LEN     = 256                # frames kept per utterance; 256 at 25 frames/s = 10.24 s
ENCODE_BATCH_SIZE = 8                  # a batch that does not fit the GPU is retried clip by clip

HELD_OUT_VOICES = 2
# GOOGLE DRIVE -- Colab only

MIRROR_TO_DRIVE = True
DRIVE_FOLDER    = "3DVGA"        # under MyDrive/
WAVS_ON_DRIVE   = True           # keep the WAVs on Drive, so a recycled runtime resumes

QUICK_SCENES = 2

## 2 — environment

Probes the GPU by launching a kernel — `torch.cuda.is_available()` alone can report `True` on an unsupported card.

In [ ]:
import glob as _glob
import importlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import textwrap
import time
from pathlib import Path

REPO = Path.cwd()
RESULTS = {}

if not (REPO / "scripts" / "audio_train.py").is_file():
    raise SystemExit(f"Run this notebook from the repository root. Current cwd: {REPO}")

# lib/config.py lists this directory at import time; an empty one is enough for the steps
# that never read a scan.
(REPO / "data" / "scannet" / "scans").mkdir(parents=True, exist_ok=True)

IN_COLAB = importlib.util.find_spec("google.colab") is not None

GPU_USABLE, GPU_NAME, GPU_NOTE = False, None, ""
try:
    import torch
    if torch.cuda.is_available():
        GPU_NAME = torch.cuda.get_device_name(0)
        major, minor = torch.cuda.get_device_capability(0)
        try:
            _probe = torch.randn(8, 8, device="cuda")
            (_probe @ _probe).sum().item()
            GPU_USABLE = True
            GPU_NOTE = f"{GPU_NAME} (sm_{major}{minor})"
        except Exception as error:
            arch = getattr(torch.cuda, "get_arch_list", lambda: ["?"])()
            GPU_NOTE = (f"{GPU_NAME} (sm_{major}{minor}) present but NO KERNEL RUNS -- "
                        f"this torch ships {arch}. "
                        f"{type(error).__name__}: {str(error).splitlines()[0][:80]}")
    else:
        GPU_NOTE = "no CUDA device visible to torch"
except Exception as error:
    GPU_NOTE = f"torch unavailable: {error}"

if DEVICE == "auto":
    DEVICE = "cuda" if GPU_USABLE else "cpu"
elif DEVICE == "cuda" and not GPU_USABLE:
    print("[warn] DEVICE='cuda' but no kernel launches here.")

print("=" * 88)
print(f"environment  : {'Google Colab' if IN_COLAB else 'local machine'}")
print(f"python       : {sys.version.split()[0]}")
print(f"GPU          : {'USABLE -- ' + GPU_NOTE if GPU_USABLE else 'UNUSABLE -- ' + GPU_NOTE}")
print(f"device       : {DEVICE}")
print(f"quick test   : {QUICK_TEST}")
print("=" * 88)

### Runner

Streams a command's output and keeps its log under `outputs/reports/<step>/`.

In [ ]:
REPORTS = REPO / "outputs" / "reports"

def run(key, cmd, env=None, note=""):
    """Run one command, stream its output, and keep a report under outputs/reports/."""
    cmd = [str(c) for c in cmd]
    printable = " ".join(cmd)
    print("=" * 88)
    print(f"STEP   {key}" + (f"\n       {note}" if note else ""))
    print(f"$ {printable}\n{'-' * 88}")

    full_env = os.environ.copy()
    full_env.update({k: str(v) for k, v in (env or {}).items()})
    full_env["CUDA_VISIBLE_DEVICES"] = GPU_ID if (DEVICE == "cuda" and GPU_USABLE) else ""
    full_env["PYTHONUNBUFFERED"] = "1"

    started = time.time()
    captured = []
    process = subprocess.Popen(cmd, cwd=str(REPO), env=full_env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
        captured.append(line)
    code = process.wait()
    elapsed = round(time.time() - started, 1)

    folder = REPORTS / key
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "log.txt").write_text("".join(captured))
    record = {"key": key, "note": note, "cmd": printable, "exit_code": code,
              "elapsed_sec": elapsed, "finished_at": time.strftime("%Y-%m-%d %H:%M:%S"),
              "environment": "Google Colab" if IN_COLAB else "local", "gpu": GPU_NAME or "none"}
    (folder / "result.json").write_text(json.dumps(record, indent=2))
    RESULTS[key] = record

    if code in (247, -9, 137):
        verdict = f"FAILED (exit {code}) -- killed by the OS, out of host RAM"
    else:
        verdict = "OK" if code == 0 else f"FAILED (exit {code})"
    print(f"{'-' * 88}\n[{key}] {verdict} in {elapsed}s   log -> {folder.relative_to(REPO)}/log.txt")
    return code

## 3 — packages

Colab only.

In [ ]:
EXTRA_PACKAGES = ["transformers", "h5py"]
REQUIRED_MODULES = ["torch", "numpy", "h5py", "transformers", "easydict"]

def _pip(*args):
    print("$ pip install " + " ".join(args))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)

def ensure_packages(force=False):
    if not (IN_COLAB or force):
        print("not on Colab -- skipping installs. Pass force=True to run them anyway.")
        return
    if (REPO / "req" / "requirements.txt").is_file():
        _pip("-r", str(REPO / "req" / "requirements.txt"))
    _pip(*EXTRA_PACKAGES)

ensure_packages()

print("\n-- packages --")
MISSING_PACKAGES = []
for module in REQUIRED_MODULES:
    try:
        loaded = importlib.import_module(module)
        print(f"  {module:18s} OK    {getattr(loaded, '__version__', '')}")
    except Exception:
        MISSING_PACKAGES.append(module)
        print(f"  {module:18s} --    MISSING")
if MISSING_PACKAGES:
    print(f"\n  install first: pip install {' '.join(MISSING_PACKAGES)}")

## 4 — Google Drive and paths

On Colab the WAVs are read from Drive and the feature files are copied there when encoding ends. Locally everything stays under `data_audio/`.

In [ ]:
# Off Colab this whole cell is inert, so the notebook stays runnable locally.

DRIVE_ROOT = None

def mount_drive():
    """Mount Drive and return the destination directory, or None if unavailable."""
    global DRIVE_ROOT
    if not (MIRROR_TO_DRIVE and IN_COLAB):
        print("Drive off"
              + ("" if MIRROR_TO_DRIVE else " (MIRROR_TO_DRIVE=False)")
              + ("" if IN_COLAB else " -- not running on Colab, everything stays local"))
        return None
    try:
        from google.colab import drive
        mount = Path("/content/drive")
        if not (mount / "MyDrive").is_dir():
            drive.mount(str(mount))
        DRIVE_ROOT = mount / "MyDrive" / DRIVE_FOLDER
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        print(f"Drive mounted -> {DRIVE_ROOT}")
    except Exception as error:
        # A failed mount must not take the run with it.
        DRIVE_ROOT = None
        print(f"Drive not mounted ({error}). The run continues; results stay on this "
              f"machine only and will be lost when the runtime recycles.")
    return DRIVE_ROOT

def copy_newer(source, target, quiet=False):
    """Copy a file or a tree, skipping files the target already has up to date."""
    source, target = Path(source), Path(target)
    if not source.exists():
        return 0
    files = [source] if source.is_file() else [p for p in source.rglob("*") if p.is_file()]
    copied = 0
    for path in files:
        if "tensorboard" in path.parts or path.suffix == ".tmp":
            continue
        dest = target if source.is_file() else target / path.relative_to(source)
        try:
            if dest.exists() and dest.stat().st_mtime >= path.stat().st_mtime \
                    and dest.stat().st_size == path.stat().st_size:
                continue
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, dest)
            copied += 1
        except Exception as error:
            if not quiet:
                print(f"  could not copy {path.name}: {error}")
    return copied

def mirror_to_drive(quiet=False):
    """Copy outputs/ to Drive. Cheap to call repeatedly: only newer files are copied."""
    if DRIVE_ROOT is None:
        return 0
    copied = copy_newer(REPO / "outputs", DRIVE_ROOT / "outputs", quiet=quiet)
    if not quiet:
        print(f"mirrored {copied} new/changed file(s) to {DRIVE_ROOT}/outputs")
    return copied

def restore_from_drive(*relative):
    """Bring files back from Drive after a fresh runtime, where they are missing here."""
    if DRIVE_ROOT is None:
        return
    for rel in relative:
        copied = copy_newer(DRIVE_ROOT / rel, REPO / rel, quiet=True)
        if copied:
            print(f"restored {copied} file(s) from Drive: {rel}")

mount_drive()

LOCAL_ROOT = REPO / "data_audio" / ("_quick" if QUICK_TEST else "")
if DRIVE_ROOT is not None and WAVS_ON_DRIVE and not QUICK_TEST:
    AUDIO_ROOT = DRIVE_ROOT / "data_audio"
else:
    AUDIO_ROOT = LOCAL_ROOT
FEAT_DIR = LOCAL_ROOT / "feat"           # always local: training reads it at every step

DATA_ENV = {
    "VGA_AUDIO_ROOT": AUDIO_ROOT,
    "VGA_AUDIO_FEAT_DIR": FEAT_DIR,
    "VGA_HELD_OUT_VOICES": HELD_OUT_VOICES,
    "VGA_ENCODER_LAYER": ENCODER_LAYER,
    "VGA_MAX_AUDIO_LEN": MAX_AUDIO_LEN,
}
SUBSET = ["--num_scenes", QUICK_SCENES] if QUICK_TEST else []

if not QUICK_TEST:
    restore_from_drive("data_audio/feat", "data_audio/manifest_train.json",
                       "data_audio/manifest_val.json")

print(f"WAVs and manifests : {AUDIO_ROOT}")
print(f"feature caches     : {FEAT_DIR}")

## 5 — preflight

What is on disk and what each stage still has to do. Changes nothing.

In [ ]:
def annotations(split):
    data = json.loads((REPO / "data" / f"ScanRefer_filtered_{split}.json").read_text())
    if QUICK_TEST:
        scenes = set(sorted({d["scene_id"] for d in data})[:QUICK_SCENES])
        data = [d for d in data if d["scene_id"] in scenes]
    return data

def manifest(split):
    path = AUDIO_ROOT / f"manifest_{split}.json"
    return json.loads(path.read_text()) if path.is_file() else None

def cache_path(split):
    return FEAT_DIR / ENCODER / f"{split}.h5"

def cached_keys(split):
    if not cache_path(split).is_file():
        return set()
    import h5py
    with h5py.File(cache_path(split), "r") as handle:
        return {k.decode("utf-8") for k in handle["keys"][:]}

def key_of(d):
    return f"{d['scene_id']}/{d['object_id']}/{d['ann_id']}"

print(f"{'split':<7} {'descriptions':>13} {'WAVs (manifest)':>16} {'in feature cache':>17}")
print("-" * 58)
TODO = {}
for split in SPLITS:
    anns = annotations(split)
    wanted = {key_of(d) for d in anns}
    listed = manifest(split)
    spoken = {e["key"] for e in (listed or {}).get("entries", []) if "error" not in e} & wanted
    encoded = cached_keys(split) & wanted
    TODO[split] = (len(wanted) - len(spoken), len(wanted) - len(encoded))
    print(f"{split:<7} {len(wanted):>13} {len(spoken):>16} {len(encoded):>17}")

to_speak = sum(t[0] for t in TODO.values())
to_encode = sum(t[1] for t in TODO.values())
print(f"\n{to_speak} description(s) still have no WAV; {to_encode} to encode.")
if to_speak:
    print("Descriptions without a WAV cannot be encoded -- run run_audio_tts.ipynb first.")

## 6 — encode

Written to a temporary file and renamed at the end, so an interrupted run leaves no half-built cache.

In [ ]:
# switch: RUN_ENCODE
if RUN_ENCODE:
    cmd = [sys.executable, "scripts/audio_encode/cache_audio_features.py", "--splits", *SPLITS,
           "--encoder", ENCODER, "--layer", ENCODER_LAYER,
           "--batch_size", ENCODE_BATCH_SIZE, *SUBSET]
    code_ = run("audio-encode", cmd, env=DATA_ENV, note="WAVs -> frozen encoder features")
    if code_ == 0 and DRIVE_ROOT is not None and not QUICK_TEST:
        copied = copy_newer(FEAT_DIR, DRIVE_ROOT / "data_audio" / "feat")
        print(f"copied {copied} feature file(s) to {DRIVE_ROOT}/data_audio/feat")
else:
    print("RUN_ENCODE is off")

## 7 — verify

Every description should have features. An utterance missing from the cache is heard as silence in training, so this must read `READY` before a real run.

In [ ]:
import numpy as np
import h5py

ready = True
for split in SPLITS:
    path = cache_path(split)
    if not path.is_file():
        print(f"{split}: no feature cache at {path}")
        ready = False
        continue
    wanted = {key_of(d) for d in annotations(split)}
    with h5py.File(path, "r") as handle:
        keys = {k.decode("utf-8") for k in handle["keys"][:]}
        lengths = np.diff(handle["offsets"][:])
        attrs = dict(handle.attrs)
        prosody = handle["prosody"][: min(20000, handle["prosody"].shape[0])]
    missing = wanted - keys
    ready &= not missing
    print(f"\n{split}: {path.name}  {path.stat().st_size / 1e9:.2f} GB")
    print(f"  encoder {attrs['encoder']} layer {attrs['encoder_layer']}, "
          f"{attrs['fps']:.0f} frames/s, dim {attrs['feat_dim']}, cap {attrs['max_audio_len']}")
    print(f"  utterances {len(keys & wanted)} of {len(wanted)}   missing {len(missing)}")
    print(f"  frames per utterance: mean {lengths.mean():.1f}  max {lengths.max()}  "
          f"at the cap {100 * (lengths >= attrs['max_audio_len']).mean():.1f}%")
    print(f"  prosody channel std: {np.round(prosody.std(axis=0), 2)}  (a zero means a dead channel)")

print("\n" + ("READY -- continue with run_audio_train_eval.ipynb" if ready
              else "NOT READY -- some descriptions have no features; re-run the stages above"))